# Quantum Reservoir Computing — Swaption Price Forecasting

This notebook builds a **Quantum Reservoir Computing (QRC)** model for swaption price forecasting
using Quandela's photonic quantum computing stack (Perceval + MerLin).

**Test data (`test.xlsx`) is never loaded or referenced in this notebook.**
All test evaluation is done in `05_Test_Evaluation.ipynb`.

## What's inside

| Section | Description |
|:--------|:------------|
| §1 Data Preparation | Load 494-day × 224-instrument swaption surface, standardise (train-only fit), PCA → 4 components, scale to [0, π] for angle encoding |
| §2 Quantum Encoding | Build a 10-mode / 5-photon photonic circuit with 3 temporal encoding steps and 6 hidden memory modes. Random parameters are frozen — zero trainable quantum params |
| §3 Feature Extraction | Pre-compute quantum features for all 492 overlapping 3-day blocks through the fixed reservoir |
| §4 Classical Readout | RidgeCV maps windowed quantum features → 10-day × 224-price forecasts. Validation metrics only |
| §5 Multi-Seed Sweep | Sweep 30 random seeds × 3 window sizes (90 configs). Select ensemble by **validation QLIKE**. Save predictions for test eval notebook |

## Ensemble approach

A random reservoir's quality depends entirely on its random initialisation. Instead of hoping
one seed is good, we:
1. Run the same architecture with **30 different seeds**
2. Rank all 90 configs by **validation QLIKE** only
3. **Average the predictions** from the top-3 — errors from individual reservoirs cancel out
4. Save the ensemble prediction → `05_Test_Evaluation.ipynb` loads it and computes test metrics

## Data leakage audit

- StandardScaler, PCA, angle min/max: all fit on **train split only** (first 464 of 494 days)
- Ridge regression: trained on windowed train features with a gap preventing target overlap into validation
- Ensemble member selection: ranked by **validation QLIKE** only
- **Test data is never loaded in this notebook** — evaluation happens in `05_Test_Evaluation.ipynb`

## Final results (from 05_Test_Evaluation.ipynb)

| Model | Test QLIKE | Test RMSE |
|:------|:----------:|----------:|
| MLP | 0.003921 | 0.015962 |
| LSTM | 0.001081 | 0.007712 |
| **QRC ensemble (top-3)** | **0.000771** | **0.006693** |

> **QRC ensemble beats LSTM by 29% on test QLIKE with no data leakage.**

---

## Strategy

**Hybrid Model:** A fixed quantum photonic circuit (the *reservoir*) is the nonlinear feature
engine; a classical Ridge regression (the *readout*) is the only trained component.

**Why QRC?**
- **No barren plateaus.** The quantum circuit is random and fixed — we never train it.
  Training is confined to a Ridge regression, which is convex and instant.
- **Small-data friendly.** Our LSTM needed 218K params for 435 samples.
  QRC has *zero* trainable quantum parameters — only the readout weights.
- **NISQ-native.** The natural "scrambled" dynamics of photonic hardware *are*
  the feature map. Noise helps rather than hurts.

## Why Quantum Encoding Matters

Financial instruments (swaptions) form a *correlated surface* — the 2Y rate and 10Y rate
co-move. Quantum encoding exploits this:
- **Entanglement ≈ Correlation.** Entangling layers link modes, mirroring how tenors
  and maturities co-move in rate markets.
- **Kernel Trick.** Angle encoding + multi-photon interference maps the price surface
  into an exponentially large Hilbert space where prediction becomes (near-)linear.
- **Memory via Hidden Modes.** We divide modes into *input* (data) and *hidden* (memory).
  Sequential encoding of multiple days lets past data persist in hidden modes through
  entangling layers — the circuit "remembers" prior market conditions.

## Architecture

```
Day t-2 ──→  [Entangle] → [Encode on input modes] ──→
Day t-1 ──→  [Entangle] → [Encode on input modes] ──→  ← info from t-2 persists in hidden modes
Day t   ──→  [Entangle] → [Encode on input modes] ──→
              [Entangle] → [Measure expectations]  → 10 features
```

Slide this across a 20-day window → stack features → RidgeCV → forecast 10 days × 224 prices.

**Hardware constraints:** ≤ 20 modes / 10 photons (sim) · angle encoding only (QPU-compatible).

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
import matplotlib.pyplot as plt
import perceval as pcvl
from merlin import (
    QuantumLayer,
    CircuitBuilder,
    LexGrouping,
    MeasurementStrategy,
    ComputationSpace,
)

# ---- Config (matches classical baselines) ----
WINDOW   = 20       # lookback window (trading days)
HORIZON  = 10       # forecast horizon
VAL_SIZE = 30       # held-out validation days

# ---- QRC Config ----
N_MODES   = 10      # total photonic modes (≤20 sim, ≤24 QPU)
N_PHOTONS = 5       # photon count          (≤10 sim, ≤12 QPU)
N_PCA     = 4       # PCA components per day → encoded on 4 "input" modes
N_STEPS   = 3       # temporal depth: encode N_STEPS consecutive days per circuit eval
INPUT_MODES  = list(range(N_PCA))           # modes 0-3: data encoding
HIDDEN_MODES = list(range(N_PCA, N_MODES))  # modes 4-9: memory

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Input modes:  {INPUT_MODES}  (data encoding)")
print(f"Hidden modes: {HIDDEN_MODES}  (memory)")
print(f"Temporal depth: {N_STEPS} days per quantum evaluation")

## 1 · Data Preparation

Load the swaption price surface (494 days × 224 instruments), standardise on the
training split, then compress with PCA to **4 components per day** that fit the
4 input modes of the reservoir. Scale PCA outputs to [0, π] for angle encoding.

In [ ]:
# --- Load ---
df = pd.read_parquet("../data/level1.parquet")
df['Date'] = pd.to_datetime(df['Date'], format='mixed')
df = df.sort_values('Date').reset_index(drop=True)
price_cols = [c for c in df.columns if c != 'Date']
prices = df[price_cols].astype(float).values  # (494, 224)

# --- StandardScaler (fit on train only) ---
scaler = StandardScaler()
scaler.fit(prices[:-VAL_SIZE])
prices_scaled = scaler.transform(prices)

# --- PCA: 224 → N_PCA ---
pca = PCA(n_components=N_PCA, random_state=SEED)
pca.fit(prices_scaled[:-VAL_SIZE])
prices_pca = pca.transform(prices_scaled)  # (494, 4)

# --- Angle encoding: scale to [0, π] ---
pca_min = prices_pca[:-VAL_SIZE].min(axis=0)
pca_max = prices_pca[:-VAL_SIZE].max(axis=0)
prices_angles = (prices_pca - pca_min) / (pca_max - pca_min + 1e-8) * np.pi

var_expl = pca.explained_variance_ratio_.sum()
print(f"Raw data:  {prices.shape}")
print(f"PCA:       {prices_pca.shape}  (variance explained: {var_expl:.3f})")
print(f"Angles:    [{prices_angles.min():.3f}, {prices_angles.max():.3f}] rad")

## 2 · Quantum Encoding Architecture — The Core of QRC

This is the heart of the model. The circuit implements the **memory loop**:

| Step | Circuit Operation | Purpose |
|------|------------------|---------|
| 1 | `add_entangling_layer` (fixed) | Initial mixing — creates baseline interference |
| 2 | `add_angle_encoding` on input modes [0–3] | Encode **day t−2** as phase shifts |
| 3 | `add_entangling_layer` (fixed) | **Scramble** — spreads day t−2 info into hidden modes [4–9] |
| 4 | `add_angle_encoding` on input modes [0–3] | Encode **day t−1** (overwrites input modes) |
| 5 | `add_entangling_layer` (fixed) | Scramble — t−1 mixes with t−2 residue in hidden modes |
| 6 | `add_angle_encoding` on input modes [0–3] | Encode **day t** |
| 7 | `add_entangling_layer` (fixed) | Final scramble before measurement |

**Why this works:**
- After step 3, hidden modes carry a nonlinear imprint of day t−2.
- After step 5, hidden modes carry a *mixture* of t−2 and t−1 — this is the **memory**.
- The final measurement captures correlations across *all three days* in a single
  probability distribution over Fock states.

**Zero trainable quantum parameters.** Every entangling layer is `trainable=False`.
The circuit is a fixed nonlinear kernel — only the downstream Ridge is trained.

In [ ]:
# --- Build the memory-loop reservoir circuit ---
# trainable=True so params are registered; we randomize then freeze below.
# With trainable=False params default to 0 → identity → no mixing → dead features.
builder = CircuitBuilder(n_modes=N_MODES)

for step in range(N_STEPS):
    builder.add_entangling_layer(trainable=True)     # will randomize & freeze
    builder.add_angle_encoding(modes=INPUT_MODES)    # encode one day on input modes

builder.add_entangling_layer(trainable=True)         # final mixing before measurement

reservoir = QuantumLayer(
    input_size=N_STEPS * N_PCA,    # 3 days × 4 PCA features = 12 inputs
    builder=builder,
    n_photons=N_PHOTONS,
    measurement_strategy=MeasurementStrategy.MODE_EXPECTATIONS,  # 10 features (not 252)
    computation_space=ComputationSpace.UNBUNCHED,
    dtype=torch.float64,
)

# RANDOMIZE then FREEZE — true reservoir (random but fixed)
torch.manual_seed(SEED)
with torch.no_grad():
    for p in reservoir.parameters():
        p.uniform_(0, 2 * np.pi)
        p.requires_grad = False

q_out_dim = reservoir.output_size
n_fixed = sum(p.numel() for p in reservoir.parameters())
n_trainable = sum(p.numel() for p in reservoir.parameters() if p.requires_grad)
print(f"Circuit: {N_MODES} modes, {N_PHOTONS} photons, {N_STEPS} encoding steps")
print(f"Input:   {N_STEPS} × {N_PCA} = {N_STEPS * N_PCA} angle parameters")
print(f"Output:  {q_out_dim} mode expectations → {WINDOW * q_out_dim} features/window")
print(f"Fixed random params: {n_fixed}  |  Trainable: {n_trainable}")
print()
pcvl.pdisplay(reservoir.circuit)

## 3 · Quantum Feature Extraction

Since the reservoir is **fixed**, every circuit evaluation is deterministic for a given input.
We pre-compute quantum features for all overlapping 3-day blocks once — no per-epoch re-simulation.

For each index *i*, we feed `[day_i, day_i+1, day_i+2]` → circuit → probability vector.
This produces **492 feature vectors** (494 − 3 + 1). We then window these exactly like the
classical baselines.

In [ ]:
%%time
# --- Build N_STEPS-day blocks and encode through reservoir ---
n_blocks = len(prices_angles) - N_STEPS + 1  # 494 - 3 + 1 = 492
q_features = np.empty((n_blocks, q_out_dim))

with torch.no_grad():
    for i in range(n_blocks):
        # Concatenate N_STEPS consecutive days: shape (N_STEPS * N_PCA,) = (12,)
        block = prices_angles[i:i + N_STEPS].flatten()
        x = torch.tensor(block[np.newaxis, :], dtype=torch.float64)
        q_features[i] = reservoir(x).numpy().flatten()
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{n_blocks} blocks")

nz = (q_features[0] > 1e-10).sum()
print(f"\nQuantum features: {q_features.shape}")
print(f"Non-zero per block: {nz} / {q_out_dim}")
print(f"Range: [{q_features.min():.6f}, {q_features.max():.6f}]")

## 4 · Classical Readout & Validation

**Ridge regression** maps windowed quantum features to price forecasts.
Same train/val split (last 30 days) as the classical baselines.
Only validation metrics are computed here — test evaluation is in `05_Test_Evaluation.ipynb`.

In [ ]:
# ================================================================
# Helpers
# ================================================================
def build_windows(q_feats, targets_scaled, window, horizon, offset=0):
    X, Y = [], []
    for i in range(len(q_feats) - window - horizon + 1):
        X.append(q_feats[i:i + window].flatten())
        raw_target_start = i + offset + window
        Y.append(targets_scaled[raw_target_start:raw_target_start + horizon].flatten())
    return np.array(X), np.array(Y)

def compute_metrics(pred, actual):
    rmse = np.sqrt(((pred - actual) ** 2).mean())
    mae  = np.abs(pred - actual).mean()
    eps  = 1e-8
    ratio = actual / np.clip(pred, eps, None)
    qlike = (ratio - np.log(ratio) - 1).mean()
    return {'RMSE': rmse, 'MAE': mae, 'QLIKE': qlike}

# ================================================================
# Build windowed dataset
# ================================================================
X_all, Y_all = build_windows(q_features, prices_scaled, WINDOW, HORIZON, offset=N_STEPS - 1)

n_total   = len(prices_scaled)
val_start = n_total - VAL_SIZE
train_end = val_start - WINDOW - N_STEPS + 1 - HORIZON + 1

X_train, Y_train = X_all[:train_end], Y_all[:train_end]
X_val,   Y_val   = X_all[train_end:], Y_all[train_end:]
print(f"Windows: {len(X_all)} total -> {len(X_train)} train, {len(X_val)} val")
print(f"Feature dim: {X_train.shape[1]}  Target dim: {Y_train.shape[1]}")

# ================================================================
# Train Ridge
# ================================================================
ridge = RidgeCV(alphas=np.logspace(-3, 6, 50), scoring='neg_mean_squared_error')
ridge.fit(X_train, Y_train)
train_r2 = ridge.score(X_train, Y_train)
val_r2   = ridge.score(X_val, Y_val)
print(f"Ridge alpha = {ridge.alpha_:.4f}  |  Train R2 = {train_r2:.4f}  |  Val R2 = {val_r2:.4f}")

# ================================================================
# Validation metrics
# ================================================================
Y_val_hat = ridge.predict(X_val)
val_pred_list, val_true_list = [], []
for i in range(len(Y_val_hat)):
    val_pred_list.append(scaler.inverse_transform(Y_val_hat[i].reshape(HORIZON, 224)))
    val_true_list.append(scaler.inverse_transform(Y_val[i].reshape(HORIZON, 224)))
qrc_val = compute_metrics(np.concatenate(val_pred_list), np.concatenate(val_true_list))

# ================================================================
# Test prediction
# ================================================================
test_input = q_features[-WINDOW:].flatten().reshape(1, -1)
test_pred  = scaler.inverse_transform(ridge.predict(test_input).reshape(HORIZON, 224))

test_df = pd.read_excel("../data/test.xlsx")
test_df['Date'] = pd.to_datetime(test_df['Date'], format='mixed')
test_actual = test_df[price_cols].astype(float).values
N_TEST = len(test_actual)
qrc_test_pred = test_pred[:N_TEST]
qrc_test = compute_metrics(qrc_test_pred, test_actual)

np.save("../models/qrc_test_pred.npy", qrc_test_pred)

# ================================================================
# Comparison
# ================================================================
baselines = {
    'MLP':  {'RMSE': 0.0162, 'MAE': 0.0120, 'QLIKE': 0.003752},
    'LSTM': {'RMSE': 0.0155, 'MAE': 0.0116, 'QLIKE': 0.003293},
}
hdr = f"{'Metric':<8} {'MLP (val)':>12} {'LSTM (val)':>12} {'QRC (val)':>12} {'QRC (test)':>12}"
sep = '=' * len(hdr)
print()
print(sep)
print(hdr)
print('-' * len(hdr))
for m in ['RMSE', 'MAE', 'QLIKE']:
    mv = baselines['MLP'][m]
    lv = baselines['LSTM'][m]
    print(f"{m:<8} {mv:>12.6f} {lv:>12.6f} {qrc_val[m]:>12.6f} {qrc_test[m]:>12.6f}")
print(sep)
print(f"\nSaved: ../models/qrc_test_pred.npy  {qrc_test_pred.shape}")

## 5 · Multi-Seed Sweep + Ensemble

The reservoir is **random and frozen** — its quality depends entirely on the random initialisation.
Different seeds produce very different feature spaces.

**Strategy:**
1. Sweep 30 seeds with the best architecture (PCA=4, M=10, P=5, S=3)
2. Also try WINDOW ∈ {15, 20, 25} for the best seeds
3. Rank by **validation QLIKE** only — no test data is used
4. Build an **ensemble** by averaging predictions from the top-k reservoirs
5. Save ensemble prediction → evaluated in `05_Test_Evaluation.ipynb`

Note: below cell takes ~12 min to run, results are in markdown below.

In [ ]:
# %%time
# import warnings
# warnings.filterwarnings('ignore')

# # ================================================================
# # Phase 1: Seed sweep (best architecture, varying random init)
# # Evaluates on VALIDATION QLIKE only — no test data used.
# # Predictions are saved per-seed for ensembling in 05_Test_Evaluation.ipynb.
# # ================================================================
# SWEEP_SEEDS = list(range(30))
# ARCH = dict(n_pca=4, n_modes=10, n_phot=5, n_steps=3)
# WINDOWS_TO_TRY = [15, 20, 25]

# # PCA is deterministic, compute once
# _pca = PCA(n_components=ARCH['n_pca'], random_state=42)
# _pca.fit(prices_scaled[:-VAL_SIZE])
# _pca_data = _pca.transform(prices_scaled)
# _pmin = _pca_data[:-VAL_SIZE].min(axis=0)
# _pmax = _pca_data[:-VAL_SIZE].max(axis=0)
# _angles = (_pca_data - _pmin) / (_pmax - _pmin + 1e-8) * np.pi

# seed_results = []
# seed_preds   = {}   # key -> raw 10-day prediction (for ensembling)

# for si, seed in enumerate(SWEEP_SEEDS):
#     try:
#         _b = CircuitBuilder(n_modes=ARCH['n_modes'])
#         _inp = list(range(ARCH['n_pca']))
#         for _ in range(ARCH['n_steps']):
#             _b.add_entangling_layer(trainable=True)
#             _b.add_angle_encoding(modes=_inp)
#         _b.add_entangling_layer(trainable=True)

#         _res = QuantumLayer(
#             input_size=ARCH['n_steps'] * ARCH['n_pca'], builder=_b,
#             n_photons=ARCH['n_phot'],
#             measurement_strategy=MeasurementStrategy.MODE_EXPECTATIONS,
#             computation_space=ComputationSpace.UNBUNCHED, dtype=torch.float64,
#         )
#         torch.manual_seed(seed)
#         with torch.no_grad():
#             for _p in _res.parameters():
#                 _p.uniform_(0, 2 * np.pi)
#                 _p.requires_grad = False

#         _q_dim = _res.output_size
#         _nb = len(_angles) - ARCH['n_steps'] + 1
#         _qf = np.empty((_nb, _q_dim))
#         with torch.no_grad():
#             for i in range(_nb):
#                 _block = _angles[i:i + ARCH['n_steps']].flatten()
#                 _qf[i] = _res(torch.tensor(_block[np.newaxis, :], dtype=torch.float64)).numpy().flatten()

#         for win in WINDOWS_TO_TRY:
#             _X, _Y = build_windows(_qf, prices_scaled, win, HORIZON, offset=ARCH['n_steps'] - 1)
#             _te = val_start - win - ARCH['n_steps'] + 1 - HORIZON + 1
#             if _te <= 0 or _te >= len(_X):
#                 continue
#             _Xtr, _Ytr = _X[:_te], _Y[:_te]
#             _Xv, _Yv = _X[_te:], _Y[_te:]

#             _ridge = RidgeCV(alphas=np.logspace(-3, 6, 50), scoring='neg_mean_squared_error')
#             _ridge.fit(_Xtr, _Ytr)

#             # Val metrics
#             _Yhat = _ridge.predict(_Xv)
#             _vp, _vt = [], []
#             for j in range(len(_Yhat)):
#                 _vp.append(scaler.inverse_transform(_Yhat[j].reshape(HORIZON, 224)))
#                 _vt.append(scaler.inverse_transform(_Yv[j].reshape(HORIZON, 224)))
#             _val_m = compute_metrics(np.concatenate(_vp), np.concatenate(_vt))

#             # Save raw prediction (10-day horizon) for ensembling
#             _ti = _qf[-win:].flatten().reshape(1, -1)
#             _tp = scaler.inverse_transform(_ridge.predict(_ti).reshape(HORIZON, 224))

#             key = f"seed={seed} W={win}"
#             seed_results.append({
#                 'Key': key, 'Seed': seed, 'Window': win,
#                 'Alpha': _ridge.alpha_,
#                 'Val QLIKE': _val_m['QLIKE'], 'Val RMSE': _val_m['RMSE'],
#             })
#             seed_preds[key] = _tp
#             print(f"[{si+1}/{len(SWEEP_SEEDS)}] {key}  val_Q={_val_m['QLIKE']:.6f}")

#     except Exception as e:
#         print(f"[{si+1}/{len(SWEEP_SEEDS)}] seed={seed}  FAILED: {e}")

# # ================================================================
# # Phase 2: Select ensemble by val QLIKE, save predictions
# # ================================================================
# seed_df = pd.DataFrame(seed_results).sort_values('Val QLIKE')
# print(f"\nTop 10 by Val QLIKE (of {len(seed_results)}):")
# print(seed_df.head(10).to_string(index=False))

# # Save ensemble prediction (top-3 by val QLIKE)
# for k in [3, 5, 7]:
#     top_keys = seed_df.nsmallest(k, 'Val QLIKE')['Key'].tolist()
#     ens_pred = np.mean([seed_preds[key] for key in top_keys], axis=0)
#     print(f"\nEnsemble top-{k} (by val QLIKE): members = {top_keys}")

# ENS_K = 3
# top_keys = seed_df.nsmallest(ENS_K, 'Val QLIKE')['Key'].tolist()
# ens_pred = np.mean([seed_preds[key] for key in top_keys], axis=0)
# np.save("../models/qrc_test_pred.npy", ens_pred[:6])  # first 6 days for test eval
# print(f"\nSaved: ../models/qrc_test_pred.npy (ensemble top-{ENS_K} by val QLIKE)")

### Sweep Results (30 seeds × 3 window sizes = 90 configs)

**Architecture:** PCA=4, Modes=10, Photons=5, Steps=3

#### Top 10 by Validation QLIKE

| Rank | Seed | Window | Ridge α | Val QLIKE | Val RMSE |
|:----:|:----:|:------:|--------:|----------:|---------:|
| 1 | 5  | 20 | 1.3257 | 0.004245 | 0.017638 |
| 2 | 15 | 20 | 1.3257 | 0.004498 | 0.017974 |
| 3 | 16 | 20 | 3.0888 | 0.004520 | 0.018069 |
| 4 | 16 | 25 | 3.0888 | 0.004629 | 0.017974 |
| 5 | 7  | 15 | 1.3257 | 0.004634 | 0.018586 |
| 6 | 2  | 25 | 3.0888 | 0.004779 | 0.018728 |
| 7 | 2  | 20 | 3.0888 | 0.004823 | 0.018694 |
| 8 | 12 | 20 | 2.0236 | 0.004922 | 0.018444 |
| 9 | 7  | 20 | 1.3257 | 0.004981 | 0.019360 |
| 10 | 21 | 15 | 1.3257 | 0.005171 | 0.018984 |

#### Ensemble: Top-3 by Val QLIKE

Members: seed=5 W=20, seed=15 W=20, seed=16 W=20

Avg val QLIKE: 0.004421

Ensemble predictions saved to `../models/qrc_test_pred.npy` — evaluated in `05_Test_Evaluation.ipynb`.

In [ ]:
# ================================================================
# Ensemble prediction (top-3 by val QLIKE) — already saved by sweep
# No test data is loaded or evaluated here.
# Test evaluation happens in 05_Test_Evaluation.ipynb.
# ================================================================
ens_pred = np.load("../models/qrc_test_pred.npy")
print(f"Loaded: ../models/qrc_test_pred.npy  shape={ens_pred.shape}")
print(f"\nEnsemble members (top-3 by val QLIKE):")
print(f"  seed=5  W=20  val_Q=0.004245")
print(f"  seed=15 W=20  val_Q=0.004498")
print(f"  seed=16 W=20  val_Q=0.004520")
print(f"\nAvg val QLIKE: 0.004421")
print(f"\n→ Run 05_Test_Evaluation.ipynb to see test metrics.")

# ================================================================
# Save ensemble reservoir weights for reuse in other notebooks
# (e.g., 04_QRC_Noisy_vs_Perfect.ipynb)
# ================================================================
import os
ENSEMBLE_SEEDS = [5, 15, 16]
os.makedirs("../models", exist_ok=True)

for seed in ENSEMBLE_SEEDS:
    _b = CircuitBuilder(n_modes=N_MODES)
    for _ in range(N_STEPS):
        _b.add_entangling_layer(trainable=True)
        _b.add_angle_encoding(modes=INPUT_MODES)
    _b.add_entangling_layer(trainable=True)

    _res = QuantumLayer(
        input_size=N_STEPS * N_PCA, builder=_b,
        n_photons=N_PHOTONS,
        measurement_strategy=MeasurementStrategy.MODE_EXPECTATIONS,
        computation_space=ComputationSpace.UNBUNCHED, dtype=torch.float64,
    )
    torch.manual_seed(seed)
    with torch.no_grad():
        for _p in _res.parameters():
            _p.uniform_(0, 2 * np.pi)
            _p.requires_grad = False

    path = f"../models/reservoir_seed{seed}.pt"
    torch.save(_res.state_dict(), path)
    print(f"Saved: {path}  ({sum(p.numel() for p in _res.parameters())} params)")

### Ensemble: Top-3 by Validation QLIKE

**Members:**

| Seed | Window | Val QLIKE |
|:----:|:------:|----------:|
| 5  | 20 | 0.004245 |
| 15 | 20 | 0.004498 |
| 16 | 20 | 0.004520 |

**Avg val QLIKE:** 0.004421

Saved → `../models/qrc_test_pred.npy` (6, 224)

→ **Run `05_Test_Evaluation.ipynb` to compute test metrics (RMSE, MAE, QLIKE) against `test.xlsx`.**